# Parameter Space Validation
This notebook visualizes the parameter space for large parameter sampling.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:.17g}")
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = "retina"
plt.rcParams.update({"figure.dpi": 150})
from itertools import combinations

from scipy.stats import kstest

## Load Parameter Samples Text File

In [ ]:
PARAM_FILE = Path(
    "/Users/robertxpearce/Desktop/reionization-emulator/data/param_samples/params_v6.txt"
)

names = ["zmean", "alpha", "kb", "b0"]
bounds = {
    "zmean": (7.0, 9.0),
    "alpha": (0.10, 0.90),
    "kb": (0.10, 2.0),
    "b0": (0.10, 0.80),
}

X = np.loadtxt(PARAM_FILE)
df = pd.DataFrame(X, columns=names)

df.head()

## Check Shape and Min/Max for Each Param

In [ ]:
print("Shape:", df.shape)

for n in names:
    lo, hi = bounds[n]
    mn, mx = df[n].min(), df[n].max()
    print(
        f"{n:5s} \t min={mn:.17g} \t max={mx:.17g} \t bounds=[{lo},{hi}] in_bounds={mn >= lo and mx <= hi}"
    )

## Extract Parameter Set(s)

In [ ]:
param_set = [
    11,
    255,
    342,
    384,
    507,
    531,
    609,
    651,
    714,
    742,
    746,
    755,
    788,
    832,
    835,
    911,
    938,
]  # Failed Sims
print(df.iloc[param_set].to_string())

## Histogram of Parameter Space
- Histograms should be flat for uniform coverage.

In [ ]:
plt.figure(figsize=(12, 3))

columns = ["zmean", "alpha", "kb", "b0"]
labels = [r"$\bar{z}$", r"$\alpha$", r"$k_b$", r"$b_0$"]

for i, (col, label) in enumerate(zip(columns, labels), start=1):
    plt.subplot(1, 4, i)
    plt.hist(df[col], bins=30, edgecolor="black")
    plt.title(label)
    plt.xlabel(label)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

## Pairwise Coverage
Shows the space-filling behavior checking for visible correlations or clustering.

In [ ]:
pairs = list(combinations(columns, 2))
pair_labels = list(combinations(labels, 2))

plt.figure(figsize=(12, 8))

for i, ((xcol, ycol), (xlabel, ylabel)) in enumerate(zip(pairs, pair_labels), start=1):
    plt.subplot(2, 3, i)
    plt.scatter(df[xcol], df[ycol], s=8, alpha=0.7)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{xlabel} vs {ylabel}")

plt.tight_layout()
plt.show()

## Marginal Uniformity (Kolmogorov-Smirnov Test)  
* D: The maximum absolute difference between the empirical CDF and the CDF of the uniform distribution.
* Large p-value: Consistent with Uniform
* Small p-value: Marginal Bias

In [ ]:
for n in names:
    lo, hi = bounds[n]
    u = (df[n] - lo) / (hi - lo)
    D, p = kstest(u, "uniform")
    print(f"{n:5s} KS D={D:.4f} p={p:.3f}")

## Pairwise Correlation

In [ ]:
print("Pearson:\n", df[columns].corr(method="pearson"))
print()
print("Spearman:\n", df[columns].corr(method="spearman"))